# 03_Advanced_Embeddings_ViT_BERT

**Goal:** Generate embeddings for text (S-BERT) and images (ViT) and save them as FAISS indices.

**Requirements:**
- GPU (strongly recommended)
- `project_data_clean.parquet`
- `images/` folder populated with product images

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import faiss
from sentence_transformers import SentenceTransformer
from transformers import ViTImageProcessor, ViTModel

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: GPU not available. Embedding generation will be slow.")

Using device: cuda


## 1. Load Data

In [2]:
# Load clean data
DATA_PATH = "project_data_clean.parquet"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found. Please run 01_Data_Ingestion_Cleaning_EDA.ipynb first.")

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} products.")

Loaded 76935 products.


## 2. Text Embeddings (S-BERT)

We combine `product_name`, `categories_en`, and `ingredients_text` to create a richer semantic representation.

In [3]:
# Initialize S-BERT
text_model_name = "sentence-transformers/all-MiniLM-L6-v2"
text_model = SentenceTransformer(text_model_name, device=device)

# Create combined text column
print("Creating combined text column...")
df['combined_text'] = (
    df['product_name'].fillna('') + " " + 
    df['categories_en'].fillna('') + " " + 
    df['ingredients_text'].fillna('')
).str.strip()

# Generate embeddings
print("Generating text embeddings...")
texts = df['combined_text'].tolist()
text_embeddings = text_model.encode(texts, show_progress_bar=True, batch_size=32, convert_to_numpy=True)

print(f"Text embeddings shape: {text_embeddings.shape}")

# Save embeddings
np.save("text_embeddings.npy", text_embeddings)

# Create FAISS index
d = text_embeddings.shape[1]
text_index = faiss.IndexFlatL2(d)
text_index.add(text_embeddings)
faiss.write_index(text_index, "text_search.index")
print("Saved text_search.index")

Creating combined text column...
Generating text embeddings...


Batches:   0%|          | 0/2405 [00:00<?, ?it/s]

Text embeddings shape: (76935, 384)
Saved text_search.index


## 3. Image Embeddings (ViT)

We process images in batches using a DataLoader and skip missing images.

In [4]:
# Initialize ViT
vit_model_name = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(vit_model_name)
vit_model = ViTModel.from_pretrained(vit_model_name).to(device)
vit_model.eval()

IMAGES_DIR = "../images"

# Filter DataFrame for existing images
print("Filtering for existing images...")
def get_image_path(row):
    code = row.get('code', '')
    if not code:
        return os.path.join(IMAGES_DIR, f"{row.name}.jpg") # Fallback to index if needed
    return os.path.join(IMAGES_DIR, f"{code}.jpg")

df['image_path'] = df.apply(get_image_path, axis=1)
df_images = df[df['image_path'].apply(os.path.exists)].copy()

print(f"Found {len(df_images)} images out of {len(df)} products.")

# Dataset definition
class ProductImageDataset(Dataset):
    def __init__(self, image_paths, processor):
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            # Return the pixel values directly
            return self.processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a dummy tensor or handle appropriately (though we filtered existence already)
            return torch.zeros((3, 224, 224))

# DataLoader
batch_size = 64
dataset = ProductImageDataset(df_images['image_path'].tolist(), processor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0) # num_workers=0 for Windows compatibility

image_embeddings = []
print("Generating image embeddings...")

with torch.no_grad():
    for batch in tqdm(dataloader):
        pixel_values = batch.to(device)
        outputs = vit_model(pixel_values=pixel_values)
        # Extract [CLS] token (index 0)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        image_embeddings.append(embeddings)

if len(image_embeddings) > 0:
    image_embeddings = np.concatenate(image_embeddings, axis=0)
else:
    image_embeddings = np.array([]).reshape(0, 768)

image_embeddings = image_embeddings.astype('float32')
print(f"Image embeddings shape: {image_embeddings.shape}")

# Save embeddings
np.save("image_embeddings.npy", image_embeddings)

# Save corresponding IDs (codes) to map back to products
# This is crucial since we filtered the dataframe
image_ids = df_images['code'].astype(str).values
np.save("image_ids.npy", image_ids)
print("Saved image_ids.npy")

# Create FAISS index
d_img = image_embeddings.shape[1]
image_index = faiss.IndexFlatL2(d_img)
image_index.add(image_embeddings)
faiss.write_index(image_index, "image_search.index")
print("Saved image_search.index")

Filtering for existing images...


Found 76866 images out of 76935 products.
Generating image embeddings...


  0%|          | 0/1202 [00:00<?, ?it/s]

Image embeddings shape: (76866, 768)
Saved image_ids.npy


Saved image_search.index


## 4. Verification

In [5]:
# Verify shapes
assert len(df) == text_embeddings.shape[0], "Text embeddings count mismatch"
assert len(df_images) == image_embeddings.shape[0], "Image embeddings count mismatch"
assert text_index.ntotal == len(df), "Text index count mismatch"
assert image_index.ntotal == len(df_images), "Image index count mismatch"

print("SUCCESS: Text embeddings match full dataframe.")
print(f"SUCCESS: Image embeddings match found images ({len(df_images)}).")

SUCCESS: Text embeddings match full dataframe.
SUCCESS: Image embeddings match found images (76866).
